# CausalMan: Generate the RCA intervention datasets

This notebook generates the **eight interventional datasets** used by the CausalMan root-cause analysis (RCA) benchmark: four intervention tasks, each evaluated on two compatible simulator scales. These are the abnormal/interventional regimes; pair them with the matching observational reference data from the main CausalMan release or the intervention tutorial when an RCA method requires normal-vs-abnormal inputs.

The release contract deliberately separates:

- `observed_data.csv`: variables available to an RCA method;
- `ground_truth/`: intervention labels, masks, and the interventional DAG;
- `metadata.json`: the complete provenance and validation record; and
- `manifest.csv` / `manifest.json`: an index of all eight datasets.

The simulator applies interventions inside each batch before ancestral sampling. The saved input data therefore preserves CausalMan's batch-dependent production-line mechanisms.

## 1. Environment

Run this notebook from the environment of the private CausalMan repository. It expects the repository's `causalman` module and its dependencies to be importable. The notebook does not install, clone, or fetch code automatically, so it always runs against the exact local CausalMan revision selected by the authors.

`Normal(name, mean, standard_deviation)` uses the standard deviation, not the variance.

In [ ]:
import copy
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import sympy
from IPython.display import display
from sympy.stats import Normal

import causalman
from causalman import CausalMan

print(f"Python: {platform.python_version()}")
print(f"CausalMan module: {Path(causalman.__file__).resolve()}")

## 2. Configuration

`BATCH_MULTIPLIER` controls the amount of data through CausalMan's native batching mechanism; the exact row count is simulator-dependent and is recorded after sampling. By default, all datasets use the same seed so differences within a scale are attributable to the interventions and not to a different batch/product draw. Set `INDEPENDENT_DATASET_SEEDS = True` only when independent dataset replicas are scientifically intended.

The output root is deterministic. The overwrite guard prevents a partial old run from being mixed with a new release.

In [ ]:
OUTPUT_ROOT = Path("output") / "causalman_rca"
BASE_SEED = 42
INDEPENDENT_DATASET_SEEDS = False
BATCH_MULTIPLIER = 1
PARALLELIZE = True
DEBUG_MODE = False
ALLOW_OVERWRITE = False
FAIL_ON_MISSING_VALUES = True
SAVE_INTERVENTIONAL_DAG = True

# Sample-size-aware checks for requested Normal interventions.
NORMAL_Z_SCORE = 5.0
NORMAL_MEAN_FLOOR_IN_SD = 0.02
NORMAL_STD_FLOOR_IN_SD = 0.05

print(f"Output root: {OUTPUT_ROOT.resolve()}")

## 3. Declarative intervention registry

The registry below is the single source of truth for all eight datasets. `evaluation_target=True` means that an observed-variable RCA method may be scored against that target.

### Note on `MV1_Emv = 190000`

The requested value is implemented exactly as specified. Under the intervention API demonstrated by CausalMan, a numeric scalar is a deterministic **atomic/hard intervention**. A genuinely stochastic/soft intervention would require a distribution and a positive standard deviation, which were not specified. Consequently, `MV1_Emv` is recorded as a hidden constant intervention and is not silently relabeled as stochastic.

In task S04, `MV1_Emv` affects the full SCM but is excluded from `observed_data.csv`. The two observed intervention targets remain the rankable RCA targets; the hidden intervention is retained in ground truth for analysis.

In [ ]:
TASKS = [
    {
        "task_id": "s01",
        "slug": "force_17000",
        "description": "Deterministic intervention on the T1 press-fitting force.",
        "scales": ["micro", "small"],
        "interventions": [
            {
                "variable": "PF_M1_T1_Force",
                "kind": "constant",
                "value": 17000.0,
                "expected_observable": True,
                "evaluation_target": True,
            }
        ],
    },
    {
        "task_id": "s02",
        "slug": "fmax_normal",
        "description": "Stochastic intervention on the T1 maximum press-fitting force.",
        "scales": ["micro", "small"],
        "interventions": [
            {
                "variable": "PF_M1_T1_Fmax",
                "kind": "normal",
                "mean": 18500.0,
                "std": 3000.0,
                "expected_observable": True,
                "evaluation_target": True,
            }
        ],
    },
    {
        "task_id": "s03",
        "slug": "two_fmax_normal",
        "description": "Joint stochastic intervention on the T1 and T2 maximum forces.",
        "scales": ["medium", "large"],
        "interventions": [
            {
                "variable": "PF_M1_T1_Fmax",
                "kind": "normal",
                "mean": 18500.0,
                "std": 3000.0,
                "expected_observable": True,
                "evaluation_target": True,
            },
            {
                "variable": "PF_M1_T2_Fmax",
                "kind": "normal",
                "mean": 19500.0,
                "std": 4000.0,
                "expected_observable": True,
                "evaluation_target": True,
            },
        ],
    },
    {
        "task_id": "s04",
        "slug": "mixed_with_hidden_emv",
        "description": "Two observed stochastic interventions plus a hidden constant intervention.",
        "scales": ["medium", "large"],
        "interventions": [
            {
                "variable": "MV2_DmvMax",
                "kind": "normal",
                "mean": 4.7,
                "std": 1.0,
                "expected_observable": True,
                "evaluation_target": True,
            },
            {
                "variable": "PF_M1_T1_Force",
                "kind": "normal",
                "mean": 16500.0,
                "std": 3000.0,
                "expected_observable": True,
                "evaluation_target": True,
            },
            {
                "variable": "MV1_Emv",
                "kind": "constant",
                "value": 190000.0,
                "expected_observable": False,
                "evaluation_target": False,
                "note": "Hidden atomic intervention; a stochastic scale was not specified.",
            },
        ],
    },
]

DATASET_SPECS = []
for task in TASKS:
    for scale in task["scales"]:
        spec = copy.deepcopy(task)
        spec.pop("scales")
        spec["scale"] = scale
        spec["simulator_name"] = f"causalman_{scale}"
        spec["dataset_id"] = f"rca_{task['task_id']}_{scale}_{task['slug']}"
        DATASET_SPECS.append(spec)

In [ ]:
assert len(DATASET_SPECS) == 8
assert len({spec["dataset_id"] for spec in DATASET_SPECS}) == 8
assert [(s["task_id"], s["scale"]) for s in DATASET_SPECS] == [
    ("s01", "micro"), ("s01", "small"),
    ("s02", "micro"), ("s02", "small"),
    ("s03", "medium"), ("s03", "large"),
    ("s04", "medium"), ("s04", "large"),
]

def compact_intervention(spec):
    if spec["kind"] == "constant":
        return f"{spec['variable']} = {spec['value']:g}"
    return f"{spec['variable']} ~ Normal({spec['mean']:g}, {spec['std']:g})"

registry_preview = pd.DataFrame([
    {
        "dataset_id": spec["dataset_id"],
        "simulator": spec["simulator_name"],
        "seed": BASE_SEED + i if INDEPENDENT_DATASET_SEEDS else BASE_SEED,
        "interventions": "; ".join(compact_intervention(x) for x in spec["interventions"]),
        "evaluation_targets": ", ".join(
            x["variable"] for x in spec["interventions"] if x["evaluation_target"]
        ),
        "hidden_targets": ", ".join(
            x["variable"] for x in spec["interventions"] if not x["expected_observable"]
        ),
    }
    for i, spec in enumerate(DATASET_SPECS)
])

display(registry_preview)

## 4. Saving and validation helpers

Observability is determined from the DAG's `Observable` node attribute, not from empirical variance. This is essential: a measured variable made constant by a hard intervention is still observable. Any columns returned by the simulator that are not DAG nodes (for example, batch bookkeeping) are saved separately as `row_metadata.csv` rather than treated as rankable RCA variables.

Normal interventions are checked with sample-size-aware tolerances. The notebook also checks the intervention table, hidden-variable exclusion, finite values, saved CSV shape/column order, and file hashes.

In [ ]:
def package_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return "local-or-unversioned"


def compile_interventions(intervention_specs):
    compiled = {}
    for intervention in intervention_specs:
        variable = intervention["variable"]
        if intervention["kind"] == "constant":
            compiled[variable] = intervention["value"]
        elif intervention["kind"] == "normal":
            compiled[variable] = Normal(
                variable, intervention["mean"], intervention["std"]
            )
        else:
            raise ValueError(f"Unsupported intervention kind: {intervention['kind']}")
    return compiled


def observable_flag(node_attributes):
    value = node_attributes.get("Observable", True)
    if isinstance(value, str):
        return value.strip().lower() not in {"false", "0", "no"}
    return bool(value)


def partition_columns(data, dag):
    graph_nodes = set(dag.nodes())
    observed = [
        column for column in data.columns
        if column in graph_nodes and observable_flag(dag.nodes[column])
    ]
    hidden = [
        column for column in data.columns
        if column in graph_nodes and not observable_flag(dag.nodes[column])
    ]
    non_graph = [column for column in data.columns if column not in graph_nodes]
    return observed, hidden, non_graph


def validate_sample(full_data, intervention_table, dag, spec, observed_columns):
    assert len(full_data) > 1, "The simulator returned too few rows."
    assert len(intervention_table) == len(full_data), (
        "The intervention table and generated data have different row counts."
    )
    assert len(full_data.columns) == len(set(full_data.columns)), "Duplicate data columns found."

    target_statistics = {}
    for intervention in spec["interventions"]:
        variable = intervention["variable"]
        assert variable in full_data.columns, f"Missing target column: {variable}"
        assert variable in dag.nodes, f"Missing target DAG node: {variable}"
        assert variable in intervention_table.columns, (
            f"The intervention table does not flag {variable}."
        )

        flags = pd.to_numeric(intervention_table[variable], errors="raise").to_numpy()
        assert np.all(flags == 1), f"Not every row is flagged as intervened for {variable}."

        is_observed = variable in observed_columns
        assert is_observed == intervention["expected_observable"], (
            f"Unexpected observability for {variable}: got {is_observed}."
        )
        if intervention["evaluation_target"]:
            assert is_observed, f"Evaluation target {variable} is not observable."

        values = pd.to_numeric(full_data[variable], errors="raise").to_numpy(dtype=float)
        assert np.isfinite(values).all(), f"Non-finite values in intervention target {variable}."

        if intervention["kind"] == "constant":
            expected = float(intervention["value"])
            assert np.allclose(values, expected, rtol=0.0, atol=1e-10), (
                f"Hard intervention check failed for {variable}."
            )
            target_statistics[variable] = {
                "kind": "constant",
                "expected_value": expected,
                "empirical_mean": float(np.mean(values)),
                "empirical_std": float(np.std(values, ddof=1)),
            }
        else:
            expected_mean = float(intervention["mean"])
            expected_std = float(intervention["std"])
            empirical_mean = float(np.mean(values))
            empirical_std = float(np.std(values, ddof=1))
            n = len(values)
            mean_tolerance = max(
                NORMAL_MEAN_FLOOR_IN_SD * expected_std,
                NORMAL_Z_SCORE * expected_std / math.sqrt(n),
            )
            std_tolerance = max(
                NORMAL_STD_FLOOR_IN_SD * expected_std,
                NORMAL_Z_SCORE * expected_std / math.sqrt(2 * (n - 1)),
            )
            assert abs(empirical_mean - expected_mean) <= mean_tolerance, (
                f"Normal mean check failed for {variable}: "
                f"{empirical_mean:.6g} vs {expected_mean:.6g} "
                f"(tolerance {mean_tolerance:.6g})."
            )
            assert abs(empirical_std - expected_std) <= std_tolerance, (
                f"Normal std check failed for {variable}: "
                f"{empirical_std:.6g} vs {expected_std:.6g} "
                f"(tolerance {std_tolerance:.6g})."
            )
            target_statistics[variable] = {
                "kind": "normal",
                "expected_mean": expected_mean,
                "expected_std": expected_std,
                "empirical_mean": empirical_mean,
                "empirical_std": empirical_std,
                "mean_tolerance": float(mean_tolerance),
                "std_tolerance": float(std_tolerance),
            }

    observed_data = full_data.loc[:, observed_columns]
    missing_values = int(observed_data.isna().sum().sum())
    if FAIL_ON_MISSING_VALUES:
        assert missing_values == 0, f"Observed data contain {missing_values} missing values."

    numeric = observed_data.select_dtypes(include=[np.number])
    if numeric.shape[1]:
        assert not np.isinf(numeric.to_numpy(dtype=float)).any(), (
            "Observed data contain positive or negative infinity."
        )

    process_result_zero_rates = {}
    for column in observed_columns:
        if column.endswith("ProcessResult"):
            numeric_result = pd.to_numeric(observed_data[column], errors="coerce")
            if numeric_result.notna().all():
                process_result_zero_rates[column] = float((numeric_result == 0).mean())

    return target_statistics, missing_values, process_result_zero_rates


def atomic_write_text(text, path):
    path = Path(path)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def atomic_write_json(payload, path):
    atomic_write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary_path = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def atomic_write_graphml(graph, path):
    path = Path(path)
    temporary_path = path.with_name(path.name + ".tmp")
    nx.write_graphml(graph, temporary_path)
    os.replace(temporary_path, path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def relative_to_output(path):
    return str(Path(path).relative_to(OUTPUT_ROOT))

## 5. Generate and save all eight datasets

The full simulator output is used in memory for validation, but it is **not** saved as the RCA input. This prevents latent variables from leaking into the benchmark. Simulator-internal artifacts are directed to a temporary directory and removed after the curated files have been written.

The notebook reports the prevalence of every observed column ending in `ProcessResult`; it does not assume that every requested intervention necessarily creates a high fault rate.

In [ ]:
# Release safeguards: portable paths, GraphML-safe attributes, and immediate
# cleanup of CausalMan's private/raw simulator artifacts after every sample.
def relative_to_output(path):
    return Path(path).relative_to(OUTPUT_ROOT).as_posix()


def graphml_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (str, int, float, bool)):
        return value
    if value is None:
        return "None"
    return str(value)


def atomic_write_graphml(graph, path):
    path = Path(path)
    safe_graph = graph.copy()
    safe_graph.graph.clear()
    for _, attributes in safe_graph.nodes(data=True):
        for key, value in list(attributes.items()):
            attributes[key] = graphml_value(value)
    for _, _, attributes in safe_graph.edges(data=True):
        for key, value in list(attributes.items()):
            attributes[key] = graphml_value(value)
    temporary_path = path.with_name(path.name + ".tmp")
    nx.write_graphml(safe_graph, temporary_path)
    os.replace(temporary_path, path)


_BaseCausalMan = CausalMan


class _EphemeralCausalMan:
    def __init__(self, **kwargs):
        requested_path = Path(kwargs["save_path"])
        requested_path.parent.mkdir(parents=True, exist_ok=True)
        self._temporary_directory = tempfile.TemporaryDirectory(
            prefix=requested_path.name + "_", dir=requested_path.parent
        )
        kwargs["save_path"] = self._temporary_directory.name
        self._simulator = _BaseCausalMan(**kwargs)

    @property
    def intervention_dict(self):
        return self._simulator.intervention_dict

    @intervention_dict.setter
    def intervention_dict(self, value):
        self._simulator.intervention_dict = value

    def sample(self):
        try:
            return self._simulator.sample()
        finally:
            self._temporary_directory.cleanup()


CausalMan = _EphemeralCausalMan

In [ ]:
_base_validate_sample = validate_sample


def validate_sample(full_data, intervention_table, dag, spec, observed_columns):
    result = _base_validate_sample(
        full_data, intervention_table, dag, spec, observed_columns
    )
    requested_targets = {item["variable"] for item in spec["interventions"]}
    positive_columns = set()
    for column in intervention_table.columns:
        flags = pd.to_numeric(intervention_table[column], errors="raise").to_numpy()
        if np.any(flags != 0):
            positive_columns.add(column)
    assert positive_columns == requested_targets, (
        f"Unexpected intervention mask: got {sorted(positive_columns)}, "
        f"expected {sorted(requested_targets)}."
    )
    return result

In [ ]:
if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()) and not ALLOW_OVERWRITE:
    raise FileExistsError(
        f"{OUTPUT_ROOT.resolve()} is not empty. Choose a new OUTPUT_ROOT or "
        "set ALLOW_OVERWRITE=True after verifying its contents."
    )
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

software_versions = {
    "python": platform.python_version(),
    "causalman": getattr(causalman, "__version__", package_version("causalman")),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "networkx": nx.__version__,
    "sympy": sympy.__version__,
}

manifest_rows = []
manifest_json = []
run_started_utc = datetime.now(timezone.utc).isoformat()

with tempfile.TemporaryDirectory(prefix="causalman_rca_simulator_") as simulator_tmp:
    simulator_tmp = Path(simulator_tmp)

    for dataset_index, spec in enumerate(DATASET_SPECS):
        dataset_id = spec["dataset_id"]
        seed = BASE_SEED + dataset_index if INDEPENDENT_DATASET_SEEDS else BASE_SEED
        print(f"[{dataset_index + 1}/8] Generating {dataset_id} (seed={seed})")

        simulator_save_path = simulator_tmp / dataset_id
        simulator_save_path.mkdir(parents=True, exist_ok=True)

        simulator = CausalMan(
            name=spec["simulator_name"],
            seed=seed,
            batch_multiplier=BATCH_MULTIPLIER,
            parallelize=PARALLELIZE,
            debug_mode=DEBUG_MODE,
            save_path=str(simulator_save_path),
        )
        simulator.intervention_dict = compile_interventions(spec["interventions"])
        full_data, intervention_table, _simulator_paths, interventional_dag = simulator.sample()

        observed_columns, hidden_columns, non_graph_columns = partition_columns(
            full_data, interventional_dag
        )
        assert observed_columns, f"No observable columns found for {dataset_id}."
        assert not set(observed_columns) & set(hidden_columns)

        target_statistics, missing_values, process_result_zero_rates = validate_sample(
            full_data, intervention_table, interventional_dag, spec, observed_columns
        )
        observed_data = full_data.loc[:, observed_columns].copy()
        row_metadata = full_data.loc[:, non_graph_columns].copy()

        all_targets = [x["variable"] for x in spec["interventions"]]
        evaluation_targets = [
            x["variable"] for x in spec["interventions"] if x["evaluation_target"]
        ]
        hidden_targets = [
            x["variable"] for x in spec["interventions"] if not x["expected_observable"]
        ]
        assert not set(hidden_targets) & set(observed_data.columns)

        dataset_dir = OUTPUT_ROOT / dataset_id
        ground_truth_dir = dataset_dir / "ground_truth"
        ground_truth_dir.mkdir(parents=True, exist_ok=ALLOW_OVERWRITE)

        data_path = dataset_dir / "observed_data.csv"
        observables_path = dataset_dir / "observable_variables.txt"
        metadata_path = dataset_dir / "metadata.json"
        row_metadata_path = dataset_dir / "row_metadata.csv"
        intervention_mask_path = ground_truth_dir / "intervention_mask.csv"
        interventions_path = ground_truth_dir / "interventions.json"
        dag_path = ground_truth_dir / "interventional_dag.graphml"

        atomic_write_csv(observed_data, data_path)
        atomic_write_text("\n".join(observed_columns) + "\n", observables_path)
        atomic_write_csv(intervention_table, intervention_mask_path)
        atomic_write_json(
            {
                "all_intervention_targets": all_targets,
                "evaluation_targets": evaluation_targets,
                "hidden_intervention_targets": hidden_targets,
                "interventions": spec["interventions"],
            },
            interventions_path,
        )
        if non_graph_columns:
            atomic_write_csv(row_metadata, row_metadata_path)
        if SAVE_INTERVENTIONAL_DAG:
            atomic_write_graphml(interventional_dag, dag_path)

        reloaded_data = pd.read_csv(data_path)
        assert reloaded_data.shape == observed_data.shape
        assert list(reloaded_data.columns) == observed_columns
        data_sha256 = sha256_file(data_path)

        metadata = {
            "schema_version": "1.0",
            "dataset_id": dataset_id,
            "task_id": spec["task_id"],
            "task_description": spec["description"],
            "scale": spec["scale"],
            "simulator_name": spec["simulator_name"],
            "seed": seed,
            "batch_multiplier": BATCH_MULTIPLIER,
            "n_rows": int(observed_data.shape[0]),
            "n_observed_columns": int(observed_data.shape[1]),
            "n_hidden_columns_in_full_scm_sample": int(len(hidden_columns)),
            "non_graph_row_metadata_columns": non_graph_columns,
            "missing_values_in_observed_data": missing_values,
            "all_intervention_targets": all_targets,
            "evaluation_targets": evaluation_targets,
            "hidden_intervention_targets": hidden_targets,
            "interventions": spec["interventions"],
            "target_statistics": target_statistics,
            "process_result_zero_rates": process_result_zero_rates,
            "files": {
                "observed_data": relative_to_output(data_path),
                "observable_variables": relative_to_output(observables_path),
                "row_metadata": (
                    relative_to_output(row_metadata_path) if non_graph_columns else None
                ),
                "intervention_mask": relative_to_output(intervention_mask_path),
                "interventions": relative_to_output(interventions_path),
                "interventional_dag": (
                    relative_to_output(dag_path) if SAVE_INTERVENTIONAL_DAG else None
                ),
            },
            "observed_data_sha256": data_sha256,
            "software_versions": software_versions,
            "generated_utc": datetime.now(timezone.utc).isoformat(),
        }
        atomic_write_json(metadata, metadata_path)

        manifest_record = {
            "dataset_id": dataset_id,
            "task_id": spec["task_id"],
            "scale": spec["scale"],
            "simulator_name": spec["simulator_name"],
            "seed": seed,
            "batch_multiplier": BATCH_MULTIPLIER,
            "n_rows": int(observed_data.shape[0]),
            "n_observed_columns": int(observed_data.shape[1]),
            "all_intervention_targets": all_targets,
            "evaluation_targets": evaluation_targets,
            "hidden_intervention_targets": hidden_targets,
            "observed_data": relative_to_output(data_path),
            "metadata": relative_to_output(metadata_path),
            "observed_data_sha256": data_sha256,
        }
        manifest_json.append(manifest_record)
        manifest_rows.append({
            **{k: v for k, v in manifest_record.items() if not isinstance(v, list)},
            "all_intervention_targets": json.dumps(all_targets),
            "evaluation_targets": json.dumps(evaluation_targets),
            "hidden_intervention_targets": json.dumps(hidden_targets),
        })

        print(
            f"    saved {observed_data.shape[0]:,} rows x "
            f"{observed_data.shape[1]:,} observed variables"
        )
        if process_result_zero_rates:
            print(f"    ProcessResult zero rates: {process_result_zero_rates}")

        del simulator, full_data, observed_data, row_metadata, intervention_table
        del interventional_dag, reloaded_data
        gc.collect()

manifest_frame = pd.DataFrame(manifest_rows)
assert len(manifest_frame) == 8
atomic_write_csv(manifest_frame, OUTPUT_ROOT / "manifest.csv")
atomic_write_json(
    {
        "schema_version": "1.0",
        "run_started_utc": run_started_utc,
        "run_completed_utc": datetime.now(timezone.utc).isoformat(),
        "software_versions": software_versions,
        "datasets": manifest_json,
    },
    OUTPUT_ROOT / "manifest.json",
)

In [ ]:
# Final release checks
assert len(manifest_frame) == 8
assert manifest_frame["dataset_id"].is_unique
for record in manifest_json:
    data_path = OUTPUT_ROOT / record["observed_data"]
    metadata_path = OUTPUT_ROOT / record["metadata"]
    assert data_path.is_file() and data_path.stat().st_size > 0
    assert metadata_path.is_file() and metadata_path.stat().st_size > 0
    assert sha256_file(data_path) == record["observed_data_sha256"]

print(f"Successfully generated and validated 8 datasets in: {OUTPUT_ROOT.resolve()}")
display(manifest_frame[[
    "dataset_id", "seed", "n_rows", "n_observed_columns",
    "evaluation_targets", "hidden_intervention_targets", "observed_data"
]])

## 6. Output layout

After a successful run:

```text
output/causalman_rca/
├── manifest.csv
├── manifest.json
├── rca_s01_micro_force_17000/
│   ├── observed_data.csv
│   ├── observable_variables.txt
│   ├── row_metadata.csv              # only when non-DAG columns exist
│   ├── metadata.json
│   └── ground_truth/
│       ├── interventions.json
│       ├── intervention_mask.csv
│       └── interventional_dag.graphml
└── ... seven more dataset directories ...
```

Use `observed_data.csv` as the feature table and `evaluation_targets` from the metadata as the variable-level RCA labels. Never concatenate the `ground_truth` files with the method inputs.

The notebook records outcome prevalence rather than enforcing a minimum anomaly rate. In particular, a force of 17,000 may remain within the desired range for parts of the simulated population; benchmark authors should inspect the reported `ProcessResult` zero rates before defining train/test or normal/anomalous splits.